In [1]:
# ==========================================================
# BLOCK 1: BINARY SETUP & DATA LOADING
# ==========================================================
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# 1. Global Reproducibility (Q1 Journal Requirement)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 2. GPU Setup (Safe Memory Growth)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs Detected: {len(gpus)}")
    except RuntimeError as e:
        print(e)

# 3. Directories & Constants
DATASET_ROOT = '/home/T2430514/Downloads/MargeDataset/Binary'
ANOMALY_DIR = os.path.join(DATASET_ROOT, 'Anomaly')
NORMAL_DIR = os.path.join(DATASET_ROOT, 'Normal')
PROCESSED_DATA_DIR = '/home/T2430514/Downloads/MargeDataset/Processed' 
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

FRAME_SIZE = (224, 224)
NUM_FRAMES = 16
BATCH_SIZE = 4 

# 4. Data Loading Logic
def create_dataframe():
    data = []
    # Anomaly (Label 1)
    if os.path.exists(ANOMALY_DIR):
        for video_file in os.listdir(ANOMALY_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(ANOMALY_DIR, video_file), 'bin_label': 1})
    
    # Normal (Label 0)
    if os.path.exists(NORMAL_DIR):
        for video_file in os.listdir(NORMAL_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(NORMAL_DIR, video_file), 'bin_label': 0})
                
    return pd.DataFrame(data)

all_df = create_dataframe()
print(f"Total Binary Videos: {len(all_df)}")

# 5. Stratified Split (Crucial for Imbalanced/Small Data)
train_df, temp_df = train_test_split(all_df, test_size=0.2, stratify=all_df['bin_label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['bin_label'], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# 6. Compute Class Weights
y_train = train_df['bin_label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))
print(f"Class Weights: {class_weights_dict}")

2026-05-17 23:14:30.446103: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-17 23:14:30.452468: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779038070.460618  801466 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779038070.463444  801466 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779038070.469815  801466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

GPUs Detected: 1
Total Binary Videos: 4534
Train: 3627 | Val: 453 | Test: 454
Class Weights: {0: np.float64(1.0002757859900717), 1: np.float64(0.9997243660418964)}


In [2]:
# ==========================================================
# BLOCK 2: FRAME EXTRACTION
# ==========================================================
import sys

def extract_and_save_frames(dataframe, output_dir, num_frames=NUM_FRAMES, frame_size=FRAME_SIZE):
    print(f"Processing {len(dataframe)} videos...")
    count = 0
    
    for idx, row in dataframe.iterrows():
        base_name = os.path.basename(row.path)
        # Use .npy for faster loading during training
        save_name = os.path.splitext(base_name)[0] + '.npy'
        output_path = os.path.join(output_dir, save_name)
        
        # Skip if already processed
        if os.path.exists(output_path): 
            continue

        cap = cv2.VideoCapture(row.path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames <= 0:
            cap.release()
            continue
            
        # Uniform Temporal Sampling (SOTA standard)
        frame_indices = np.linspace(0, max(total_frames - 1, 0), num=num_frames, dtype=int)
        frames = []
        
        for i in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, frame_size)
                frames.append(frame)
            else:
                # Padding if read fails
                frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
        cap.release()
        
        # Ensure exact frame count
        while len(frames) < num_frames:
            frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
            
        # Save as uint8 to save disk space (converted to float32 in generator)
        np.save(output_path, np.array(frames, dtype=np.uint8))
        
        count += 1
        if count % 100 == 0: 
            sys.stdout.write(f"\rExtracted {count} videos.")
    print("\nExtraction Complete.")

extract_and_save_frames(all_df, PROCESSED_DATA_DIR)

Processing 4534 videos...

Extraction Complete.


In [3]:
# ==========================================================
# BLOCK 3: SOTA BINARY DATA GENERATOR (CONFIGURED FOR NO-AUG)
# ==========================================================
import tensorflow as tf
import numpy as np
import os

class SOTAVideoDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, processed_data_dir, batch_size=BATCH_SIZE, 
                 num_frames=NUM_FRAMES, frame_size=FRAME_SIZE, 
                 augment=False, shuffle=True, mixup_alpha=0.0):
        self.dataframe = dataframe
        self.processed_data_dir = processed_data_dir
        self.batch_size = batch_size
        self.num_frames = num_frames
        self.frame_size = frame_size
        self.augment = augment
        self.shuffle = shuffle
        self.mixup_alpha = mixup_alpha 
        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        return self.__data_generation(batch_indices)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def load_video(self, video_path):
        base_name = os.path.basename(video_path)
        name, _ = os.path.splitext(base_name)
        npy_path = os.path.join(self.processed_data_dir, name + '.npy')
        
        if os.path.exists(npy_path):
            try: 
                # Load and Normalize to [0,1]
                return np.load(npy_path).astype(np.float32) / 255.0
            except: 
                pass
        # Return black video if load fails
        return np.zeros((self.num_frames, *self.frame_size, 3), dtype=np.float32)

    def apply_consistent_augmentation(self, video):
        """
        Applies Augmentations via TF.image. 
        Ensures operations like Flip are consistent across the time dimension.
        """
        # 1. Random Flip
        if tf.random.uniform(()) > 0.5:
            video = tf.image.flip_left_right(video)
            
        # 2. Random Brightness
        video = tf.image.random_brightness(video, max_delta=0.2)
        video = tf.image.random_contrast(video, lower=0.8, upper=1.2)
        
        # 3. Random Saturation
        video = tf.image.random_saturation(video, lower=0.8, upper=1.2)
        
        return tf.clip_by_value(video, 0.0, 1.0)

    def __data_generation(self, batch_indices):
        X = np.empty((self.batch_size, self.num_frames, *self.frame_size, 3), dtype=np.float32)
        y = np.empty((self.batch_size), dtype=np.float32) 

        for i, idx in enumerate(batch_indices):
            row = self.dataframe.iloc[idx]
            video = self.load_video(row.path)
            label = float(row.bin_label)

            # --- MIXUP LOGIC (Will trigger only if augment=True AND mixup_alpha > 0) ---
            if self.augment and self.mixup_alpha > 0 and np.random.rand() < 0.5:
                rand_idx = np.random.choice(self.indices)
                row2 = self.dataframe.iloc[rand_idx]
                
                video2 = self.load_video(row2.path)
                label2 = float(row2.bin_label)

                lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
                video = lam * video + (1 - lam) * video2
                label = lam * label + (1 - lam) * label2

            # --- SPATIAL AUGMENTATION ---
            if self.augment:
                video = self.apply_consistent_augmentation(video)

            X[i,] = video
            y[i] = label

        return X, y
    
    def get_labels(self):
        """Helper to get all labels in order (for evaluation without shuffle)."""
        original_indices = self.indices.copy()
        if self.shuffle:
            sorted_indices = np.arange(len(self.dataframe))
        else:
            sorted_indices = self.indices
            
        limit = self.__len__() * self.batch_size
        return self.dataframe.iloc[sorted_indices[:limit]]['bin_label'].values

# Initialize Generators for Experiment 1: NO AUGMENTATION
print("Initializing Generators (Experiment 1: No Augmentation)...")

# TRAIN: No Augmentation, No MixUp (Baseline)
train_generator = SOTAVideoDataGenerator(
    train_df, 
    PROCESSED_DATA_DIR, 
    batch_size=BATCH_SIZE, 
    augment=False,        # DISABLED
    mixup_alpha=0.0,      # DISABLED
    shuffle=True
)

# VAL: No Augmentation (Standard)
val_generator = SOTAVideoDataGenerator(
    val_df, 
    PROCESSED_DATA_DIR, 
    batch_size=BATCH_SIZE, 
    augment=False, 
    shuffle=False
)
print("Generators Ready.")

Initializing Generators (Experiment 1: No Augmentation)...
Generators Ready.


In [4]:
# ==========================================================
# BLOCK 4 & 5: 3D RESNET MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
clear_session()
gc.collect()

# 2. SOTA Training Configuration
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define 3D ResNet Building Blocks
def conv3d_bn(x, filters, kernel_size, strides=(1,1,1), padding='same', activation=True):
    x = Conv3D(filters, kernel_size, strides=strides, padding=padding, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    if activation:
        x = Activation('relu')(x)
    return x

def resnet_block(x, filters, strides=(1,1,1)):
    shortcut = x
    x = conv3d_bn(x, filters, kernel_size=(3,3,3), strides=strides)
    x = conv3d_bn(x, filters, kernel_size=(3,3,3), activation=False)
    if strides != (1,1,1) or x.shape[-1] != shortcut.shape[-1]:
        shortcut = conv3d_bn(shortcut, filters, kernel_size=(1,1,1), strides=strides, activation=False)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_resnet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    x = conv3d_bn(video_input, 64, kernel_size=(7,7,7), strides=(1,2,2))
    x = tf.keras.layers.MaxPooling3D(pool_size=(1,3,3), strides=(1,2,2), padding='same')(x)
    
    x = resnet_block(x, 64)
    x = resnet_block(x, 64)
    x = resnet_block(x, 128, strides=(2,2,2)) 
    x = resnet_block(x, 128)
    x = resnet_block(x, 256, strides=(2,2,2))
    x = resnet_block(x, 256)
    x = resnet_block(x, 512, strides=(2,2,2))
    x = resnet_block(x, 512)
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs=video_input, outputs=output, name='ResNet3D_18')

# 5. Initialize & Compile
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resnet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

# 6. Start Training
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 7. SOTA Evaluation & Inference Benchmarking
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

# Inference Latency/Throughput tracking
inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# Specificity (True Negative Rate)
spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

# Parameter Calculations & FP32 Model Size
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 8. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

I0000 00:00:1776533983.401277 2059844 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13665 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: ResNet3D_18...
Epoch 1/50


I0000 00:00:1776533988.206323 2077083 service.cc:152] XLA service 0x71f87c8572f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776533988.206338 2077083 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-18 23:39:48.429711: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776533989.380932 2077083 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-18 23:39:52.749279: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller b

  1/906 ━━━━━━━━━━━━━━━━━━━━ 3:28:50 14s/step - accuracy: 1.0000 - auc: 0.0000e+00 - loss: 0.3038

2026-04-18 23:39:57.640759: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion', 484 bytes spill stores, 484 bytes spill loads

I0000 00:00:1776533997.682806 2077083 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 90s 84ms/step - accuracy: 0.6992 - auc: 0.7668 - loss: 0.7218 - val_accuracy: 0.7677 - val_auc: 0.8404 - val_loss: 0.5803
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 75s 83ms/step - accuracy: 0.7522 - auc: 0.8250 - loss: 0.6230 - val_accuracy: 0.7765 - val_auc: 0.8534 - val_loss: 0.5907
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 74s 82ms/step - accuracy: 0.7892 - auc: 0.8584 - loss: 0.5747 - val_accuracy: 0.7566 - val_auc: 0.8240 - val_loss: 0.6421
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 74s 82ms/step - accuracy: 0.7999 - auc: 0.8694 - loss: 0.5614 - val_accuracy: 0.6969 - val_auc: 0.7988 - val_loss: 0.6713
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 75s 82ms/step - accuracy: 0.8193 - auc: 0.8904 - loss: 0.5299 - val_accuracy: 0.8053 - val_auc: 0.8668 - val_loss: 0.5902
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 74s 82ms/step - accuracy: 0.8228 - auc: 0.8972 - loss: 0.5176 - val_accuracy: 0.8142 - val_auc: 0.8497 - val_loss: 0.5913
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 6 & 7: I3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Concatenate,
    MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for I3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 5e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define I3D Building Blocks
# ----------------------------------------------------------
def conv3d_bn(x, filters, kernel_size, padding='same', strides=(1,1,1), name=None):
    x = Conv3D(filters, kernel_size, strides=strides, padding=padding, 
               use_bias=False, kernel_regularizer=l2(1e-5), name=name)(x)
    x = BatchNormalization(scale=False)(x)
    x = Activation('relu')(x)
    return x

def inception_module(x, filters):
    f1x1, f3x3_reduce, f3x3, f5x5_reduce, f5x5, f_pool = filters

    branch1 = conv3d_bn(x, f1x1, (1, 1, 1))
    
    branch2 = conv3d_bn(x, f3x3_reduce, (1, 1, 1))
    branch2 = conv3d_bn(branch2, f3x3, (3, 3, 3))

    branch3 = conv3d_bn(x, f5x5_reduce, (1, 1, 1))
    branch3 = conv3d_bn(branch3, f5x5, (3, 3, 3))

    branch4 = MaxPooling3D((3, 3, 3), strides=(1, 1, 1), padding='same')(x)
    branch4 = conv3d_bn(branch4, f_pool, (1, 1, 1))

    x = Concatenate()([branch1, branch2, branch3, branch4])
    return x

def create_i3d_model(input_shape):
    video_input = Input(shape=input_shape)

    x = conv3d_bn(video_input, 64, (7, 7, 7), strides=(2, 2, 2))
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)
    
    x = conv3d_bn(x, 64, (1, 1, 1))
    x = conv3d_bn(x, 192, (3, 3, 3))
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = inception_module(x, [64, 96, 128, 16, 32, 32])
    x = inception_module(x, [128, 128, 192, 32, 96, 64])
    x = MaxPooling3D((2, 2, 2), strides=(2, 2, 2), padding='same')(x)

    x = inception_module(x, [192, 96, 208, 16, 48, 64])
    x = inception_module(x, [160, 112, 224, 24, 64, 64])
    x = MaxPooling3D((2, 2, 2), strides=(2, 2, 2), padding='same')(x)

    x = inception_module(x, [128, 128, 256, 24, 64, 64])
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='I3D_Inception')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_i3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50 
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_i3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

# Parameter Calculations & FP32 Model Size
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for I3D.

Training Model: I3D_Inception...
Epoch 1/50
905/906 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6583 - auc: 0.7198 - loss: 0.7270

2026-04-19 00:44:57.111770: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1045', 96 bytes spill stores, 96 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 61s 47ms/step - accuracy: 0.6885 - auc: 0.7478 - loss: 0.7065 - val_accuracy: 0.7168 - val_auc: 0.8168 - val_loss: 0.6579
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.7337 - auc: 0.8013 - loss: 0.6526 - val_accuracy: 0.7456 - val_auc: 0.8316 - val_loss: 0.6339
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.7688 - auc: 0.8361 - loss: 0.6166 - val_accuracy: 0.7478 - val_auc: 0.8403 - val_loss: 0.6144
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.7804 - auc: 0.8453 - loss: 0.6043 - val_accuracy: 0.6947 - val_auc: 0.8095 - val_loss: 0.6993
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.7928 - auc: 0.8626 - loss: 0.5846 - val_accuracy: 0.7810 - val_auc: 0.8650 - val_loss: 0.5921
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.8085 - auc: 0.8732 - loss: 0.5712 - val_accuracy: 0.7854 - val_auc: 0.8630 - val_loss: 0.5874
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [6]:
# ==========================================================
# BLOCK 8 & 9: 3D DENSENET MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Concatenate,
    AveragePooling3D, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for 3D DenseNet.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define 3D DenseNet Building Blocks
# ----------------------------------------------------------
def conv_block(x, growth_rate, name):
    x1 = BatchNormalization(name=name+'_0_bn')(x)
    x1 = Activation('relu', name=name+'_0_relu')(x1)
    x1 = Conv3D(4 * growth_rate, (1, 1, 1), use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_1_conv')(x1)
    
    x1 = BatchNormalization(name=name+'_1_bn')(x1)
    x1 = Activation('relu', name=name+'_1_relu')(x1)
    x1 = Conv3D(growth_rate, (3, 3, 3), padding='same', use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_2_conv')(x1)
    
    x = Concatenate(name=name+'_concat')([x, x1])
    return x

def dense_block(x, blocks, name):
    # Reduced growth rate from 32 to 16 to prevent GPU OOM
    for i in range(blocks):
        x = conv_block(x, growth_rate=16, name=name + '_block' + str(i + 1))
    return x

def transition_block(x, reduction, name):
    x = BatchNormalization(name=name+'_bn')(x)
    x = Activation('relu', name=name+'_relu')(x)
    x = Conv3D(int(tf.keras.backend.int_shape(x)[-1] * reduction), (1, 1, 1), use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_conv')(x)
    x = AveragePooling3D((2, 2, 2), strides=(2, 2, 2), name=name+'_pool')(x)
    return x

def create_densenet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(64, (7, 7, 7), strides=(1, 2, 2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5), name='conv1_conv')(video_input)
    x = BatchNormalization(name='conv1_bn')(x)
    x = Activation('relu', name='conv1_relu')(x)
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same', name='pool1')(x)
    
    # Scaled down depth blocks for 3D Video memory limits
    x = dense_block(x, blocks=4, name='conv2')
    x = transition_block(x, 0.5, name='pool2')
    
    x = dense_block(x, blocks=8, name='conv3')
    x = transition_block(x, 0.5, name='pool3')
    
    x = dense_block(x, blocks=12, name='conv4')
    x = transition_block(x, 0.5, name='pool4')
    
    x = dense_block(x, blocks=8, name='conv5')
    
    x = BatchNormalization(name='bn')(x)
    x = Activation('relu')(x)
    x = GlobalAveragePooling3D(name='global_avg_pool')(x)
    
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid', name='fc_output')(x)

    return Model(inputs=video_input, outputs=output, name='DenseNet3D_Lite')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_densenet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50 
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_densenet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for 3D DenseNet.

Training Model: DenseNet3D_Lite...
Epoch 1/50


2026-04-19 01:11:40.027471: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_85', 28 bytes spill stores, 28 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 112s 65ms/step - accuracy: 0.7053 - auc: 0.7747 - loss: 0.6499 - val_accuracy: 0.7412 - val_auc: 0.8270 - val_loss: 0.6165
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.7406 - auc: 0.8136 - loss: 0.6080 - val_accuracy: 0.7279 - val_auc: 0.8309 - val_loss: 0.6002
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.7621 - auc: 0.8319 - loss: 0.5901 - val_accuracy: 0.7456 - val_auc: 0.8376 - val_loss: 0.5997
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.7646 - auc: 0.8447 - loss: 0.5753 - val_accuracy: 0.7588 - val_auc: 0.8515 - val_loss: 0.5685
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.7773 - auc: 0.8581 - loss: 0.5611 - val_accuracy: 0.6150 - val_auc: 0.8219 - val_loss: 0.7960
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.7999 - auc: 0.8747 - loss: 0.5420 - val_accuracy: 0.7633 - val_auc: 0.8427 - val_loss: 0.6168
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================================
# BLOCK 10 & 11: CONVNEXT-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, LayerNormalization, Dense, GlobalAveragePooling3D, 
    Dropout, Activation, Permute, Reshape, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ConvNeXt-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.05 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ConvNeXt-3D Building Blocks
# ----------------------------------------------------------
class ConvNeXtBlock(tf.keras.layers.Layer):
    def __init__(self, dim, drop_path=0., **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.dwconv = Conv3D(dim, kernel_size=7, padding='same', groups=dim)
        self.norm = LayerNormalization(epsilon=1e-6)
        
        self.pwconv1 = Dense(4 * dim) 
        self.act = Activation('gelu')
        
        self.pwconv2 = Dense(dim)
        self.drop_path = Dropout(drop_path) if drop_path > 0. else tf.identity

    def call(self, inputs):
        input_tensor = inputs
        x = self.dwconv(inputs)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.drop_path(x)
        return input_tensor + x

def create_convnext3d_model(input_shape, depths=[3, 3, 9, 3], dims=[64, 128, 256, 512]):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(dims[0], kernel_size=(2, 4, 4), strides=(2, 4, 4), padding='valid', name='stem_conv')(video_input)
    x = LayerNormalization(epsilon=1e-6, name='stem_ln')(x)
    
    for i in range(4):
        dim = dims[i]
        depth = depths[i]
        
        for j in range(depth):
            x = ConvNeXtBlock(dim, name=f'stage{i}_block{j}')(x)
            
        if i < 3:
            x = LayerNormalization(epsilon=1e-6, name=f'stage{i}_downsample_ln')(x)
            x = Conv3D(dims[i+1], kernel_size=2, strides=2, padding='valid', name=f'stage{i}_downsample_conv')(x)

    x = GlobalAveragePooling3D()(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='ConvNeXt3D_Nano')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_convnext3d_model(input_shape, depths=[2, 2, 6, 2], dims=[48, 96, 192, 384])

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_convnext3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ConvNeXt-3D.

Training Model: ConvNeXt3D_Nano...
Epoch 1/50


2026-04-19 01:50:34.914747: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 20 bytes spill stores, 20 bytes spill loads

2026-04-19 01:50:35.084484: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 428 bytes spill stores, 428 bytes spill loads

2026-04-19 01:50:35.112937: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 516 bytes spill stores, 516 bytes spill loads

2026-04-19 01:50:35.116572: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 428 bytes spill stores, 428 bytes spill loads

2026-04-19 01:50:35.148414: I external/local_xla/xla/s

  1/906 ━━━━━━━━━━━━━━━━━━━━ 4:45:23 19s/step - accuracy: 0.2500 - auc: 0.1667 - loss: 1.0002

2026-04-19 01:50:45.552075: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_14', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_9', 4 bytes spill stores, 4 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 62s 48ms/step - accuracy: 0.6209 - auc: 0.6567 - loss: 0.7296 - val_accuracy: 0.7080 - val_auc: 0.8004 - val_loss: 0.6194
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.7655 - auc: 0.8195 - loss: 0.5620 - val_accuracy: 0.7412 - val_auc: 0.8181 - val_loss: 0.5706
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.8126 - auc: 0.8625 - loss: 0.5115 - val_accuracy: 0.8164 - val_auc: 0.8684 - val_loss: 0.5079
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.8535 - auc: 0.8989 - loss: 0.4624 - val_accuracy: 0.8274 - val_auc: 0.8918 - val_loss: 0.4756
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.8526 - auc: 0.9136 - loss: 0.4437 - val_accuracy: 0.8296 - val_auc: 0.8974 - val_loss: 0.4740
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.8667 - auc: 0.9250 - loss: 0.4253 - val_accuracy: 0.8186 - val_auc: 0.8928 - val_loss: 0.4859
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [8]:
# ==========================================================
# BLOCK 12 & 13: RESNEXT-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Concatenate,
    MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResNeXt-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ResNeXt-3D Building Blocks
# ----------------------------------------------------------
def grouped_conv3d(x, filters, kernel_size, strides=(1,1,1), padding='same', groups=8):
    try:
        return Conv3D(filters, kernel_size, strides=strides, padding=padding, 
                      groups=groups, use_bias=False, kernel_regularizer=l2(1e-5))(x)
    except:
        group_list = []
        channels_per_group = filters // groups
        splits = tf.split(x, groups, axis=-1)
        for i in range(groups):
            g = Conv3D(channels_per_group, kernel_size, strides=strides, padding=padding, 
                       use_bias=False, kernel_regularizer=l2(1e-5))(splits[i])
            group_list.append(g)
        return Concatenate(axis=-1)(group_list)

def resnext_block(x, filters, strides=(1,1,1), groups=8):
    shortcut = x
    bottleneck_width = filters // 2 

    x = Conv3D(bottleneck_width, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = grouped_conv3d(x, bottleneck_width, (3,3,3), strides=strides, padding='same', groups=groups)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)

    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_resnext3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # Scaled down stem filters from 64 to 32
    x = Conv3D(32, (7,7,7), strides=(1,2,2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(video_input)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x)

    # Scaled down depths and set groups=8 to fit 16GB VRAM
    x = resnext_block(x, 64, groups=8)
    x = resnext_block(x, 64, groups=8)

    x = resnext_block(x, 128, strides=(2,2,2), groups=8)
    x = resnext_block(x, 128, groups=8)

    x = resnext_block(x, 256, strides=(2,2,2), groups=8)
    x = resnext_block(x, 256, groups=8)

    x = resnext_block(x, 512, strides=(2,2,2), groups=8)
    x = resnext_block(x, 512, groups=8)

    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='ResNeXt3D_Lite')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnext3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resnext3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResNeXt-3D.

Training Model: ResNeXt3D_Lite...
Epoch 1/50
  3/906 ━━━━━━━━━━━━━━━━━━━━ 33s 37ms/step - accuracy: 0.3333 - auc: 0.4444 - loss: 1.0427   

2026-04-19 02:13:31.821707: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion', 16 bytes spill stores, 16 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.6841 - auc: 0.7426 - loss: 0.7165 - val_accuracy: 0.7279 - val_auc: 0.7868 - val_loss: 0.7999
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.7254 - auc: 0.7938 - loss: 0.6518 - val_accuracy: 0.7035 - val_auc: 0.8286 - val_loss: 0.7424
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.7406 - auc: 0.8253 - loss: 0.6074 - val_accuracy: 0.7345 - val_auc: 0.8321 - val_loss: 0.6021
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.7594 - auc: 0.8409 - loss: 0.5875 - val_accuracy: 0.7522 - val_auc: 0.8548 - val_loss: 0.5735
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.7867 - auc: 0.8602 - loss: 0.5608 - val_accuracy: 0.7301 - val_auc: 0.8380 - val_loss: 0.6019
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.7980 - auc: 0.8743 - loss: 0.5433 - val_accuracy: 0.7566 - val_auc: 0.8586 - val_loss: 0.5894
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [9]:
# ==========================================================
# BLOCK 14 & 15: EFFICIENTNET-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Multiply,
    Reshape
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for EfficientNet-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-5
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define EfficientNet-3D Building Blocks
# ----------------------------------------------------------
def get_activation(activation='swish'):
    return Activation(tf.nn.swish)

def squeeze_excitation_block(x, input_channels, squeeze_ratio=0.25):
    reduced_channels = max(1, int(input_channels * squeeze_ratio))
    
    se = GlobalAveragePooling3D()(x)
    se = Reshape((1, 1, 1, input_channels))(se)
    
    se = Dense(reduced_channels, kernel_initializer='he_normal', use_bias=True)(se)
    se = get_activation('swish')(se)
    
    se = Dense(input_channels, kernel_initializer='he_normal', use_bias=True)(se)
    se = Activation('sigmoid')(se)
    
    x = Multiply()([x, se])
    return x

def mbconv_block(x, input_filters, output_filters, kernel_size, strides, expand_ratio, use_se=True, drop_rate=0.0):
    shortcut = x 
    
    expanded_filters = input_filters * expand_ratio
    if expand_ratio != 1:
        x = Conv3D(expanded_filters, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = get_activation('swish')(x)
    
    # Factorized (2+1)D approach to avoid 3D Dense Kernel explosion
    # 1. Spatial Convolution
    x = Conv3D(expanded_filters, (1, kernel_size, kernel_size), strides=(1, strides[1], strides[2]), padding='same', 
               use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    # 2. Temporal Convolution
    x = Conv3D(expanded_filters, (kernel_size, 1, 1), strides=(strides[0], 1, 1), padding='same', 
               use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    if use_se:
        x = squeeze_excitation_block(x, expanded_filters)
    
    x = Conv3D(output_filters, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    
    if strides == (1, 1, 1) and input_filters == output_filters:
        if drop_rate > 0:
            x = Dropout(drop_rate)(x)
        x = Add()([shortcut, x]) 
    
    return x

def create_efficientnet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # Stem
    x = Conv3D(16, 3, strides=(2, 2, 2), padding='same', use_bias=False, kernel_initializer='he_normal')(video_input)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    # Stage 1
    x = mbconv_block(x, 16, 8, kernel_size=3, strides=(1,1,1), expand_ratio=1)
    
    # Stage 2
    x = mbconv_block(x, 8, 16, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    
    # Stage 3
    x = mbconv_block(x, 16, 24, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    
    # Stage 4
    x = mbconv_block(x, 24, 48, kernel_size=3, strides=(1,2,2), expand_ratio=2)
    x = mbconv_block(x, 48, 64, kernel_size=3, strides=(1,1,1), expand_ratio=2)
    
    # Stage 5
    x = mbconv_block(x, 64, 96, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    x = mbconv_block(x, 96, 128, kernel_size=3, strides=(1,1,1), expand_ratio=2)
    
    # Head
    x = Conv3D(512, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.2)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='EfficientNet3D_Factorized')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_efficientnet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_efficientnet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for EfficientNet-3D.

Training Model: EfficientNet3D_Factorized...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 37s 21ms/step - accuracy: 0.7133 - auc: 0.7665 - loss: 0.6083 - val_accuracy: 0.7124 - val_auc: 0.7733 - val_loss: 1.0135
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.7365 - auc: 0.7944 - loss: 0.5819 - val_accuracy: 0.5996 - val_auc: 0.7001 - val_loss: 0.7035
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.7406 - auc: 0.8058 - loss: 0.5706 - val_accuracy: 0.7212 - val_auc: 0.8062 - val_loss: 0.6188
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.7597 - auc: 0.8256 - loss: 0.5553 - val_accuracy: 0.7367 - val_auc: 0.8180 - val_loss: 0.5586
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.7666 - auc: 0.8453 - loss: 0.5322 - val_accuracy: 0.7500 - val_auc: 0.8182 - val_loss: 0.5598
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.7735 - auc: 0.8422 - loss: 0.5345 - 

In [10]:
# ==========================================================
# BLOCK 16 & 17: VIVIT MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ViViT.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.01 
LABEL_SMOOTHING = 0.1 
PROJECTION_DIM = 64  
NUM_HEADS = 4
TRANSFORMER_LAYERS = 4

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ViViT Components
# ----------------------------------------------------------
class TubeletEmbedding(layers.Layer):
    def __init__(self, embed_dim, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.projection = layers.Conv3D(
            filters=embed_dim, kernel_size=patch_size,
            strides=patch_size, padding="VALID", name="tubelet_proj"
        )
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, videos):
        projected_patches = self.projection(videos)
        flattened_patches = self.flatten(projected_patches)
        return flattened_patches

class PositionalEncoder(layers.Layer):
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        _, num_tokens, _ = input_shape
        self.position_embedding = self.add_weight(
            name="pos_embedding", shape=(num_tokens, self.embed_dim),
            initializer="glorot_uniform", trainable=True
        )

    def call(self, encoded_tokens):
        return encoded_tokens + self.position_embedding

def transformer_encoder_block(inputs, embed_dim, num_heads, ff_dim, dropout=0.1):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=embed_dim, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Add()([x, inputs]) 

    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(ff_dim, activation=tf.nn.gelu)(y) 
    y = layers.Dropout(dropout)(y)
    y = layers.Dense(embed_dim)(y)
    y = layers.Add()([y, x]) 
    return y

def create_vivit_model(input_shape):
    video_input = layers.Input(shape=input_shape)

    patches = TubeletEmbedding(embed_dim=PROJECTION_DIM, patch_size=(2, 16, 16))(video_input)
    encoded_patches = PositionalEncoder(embed_dim=PROJECTION_DIM)(patches)

    for _ in range(TRANSFORMER_LAYERS):
        encoded_patches = transformer_encoder_block(
            encoded_patches, embed_dim=PROJECTION_DIM, 
            num_heads=NUM_HEADS, ff_dim=PROJECTION_DIM * 2, dropout=0.1
        )

    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.GlobalAveragePooling1D()(representation)
    
    x = layers.Dropout(0.5)(representation)
    x = layers.Dense(128, activation=tf.nn.gelu)(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    return Model(inputs=video_input, outputs=output, name="ViViT_Transformer")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_vivit_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_vivit_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ViViT.

Training Model: ViViT_Transformer...
Epoch 1/50


2026-04-19 02:49:36.734694: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_103', 212 bytes spill stores, 212 bytes spill loads

2026-04-19 02:49:36.783039: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_113', 348 bytes spill stores, 348 bytes spill loads

2026-04-19 02:49:36.861145: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_96', 60 bytes spill stores, 60 bytes spill loads

2026-04-19 02:49:36.869142: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_96', 60 bytes spill stores, 60 bytes spill loads

2026-04-19 02:49:36.893086: I external/loc

905/906 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.4918 - auc: 0.4885 - loss: 0.8142

2026-04-19 02:50:10.904501: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1286', 8 bytes spill stores, 8 bytes spill loads

2026-04-19 02:50:10.924457: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_38', 512 bytes spill stores, 512 bytes spill loads

2026-04-19 02:50:10.991361: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 192 bytes spill stores, 192 bytes spill loads

2026-04-19 02:50:10.997399: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_38', 420 bytes spill stores, 420 bytes spill loads

2026-04-19 02:50:11.013007: I external/loc

906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 34ms/step - accuracy: 0.5132 - auc: 0.5183 - loss: 0.7693 - val_accuracy: 0.5354 - val_auc: 0.5912 - val_loss: 0.6909
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 28s 31ms/step - accuracy: 0.5337 - auc: 0.5476 - loss: 0.7217 - val_accuracy: 0.5575 - val_auc: 0.6116 - val_loss: 0.6772
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 29s 31ms/step - accuracy: 0.5579 - auc: 0.5740 - loss: 0.7026 - val_accuracy: 0.5398 - val_auc: 0.6227 - val_loss: 0.6823
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 29s 32ms/step - accuracy: 0.5671 - auc: 0.5984 - loss: 0.6905 - val_accuracy: 0.5619 - val_auc: 0.6534 - val_loss: 0.6655
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 28s 31ms/step - accuracy: 0.6026 - auc: 0.6460 - loss: 0.6672 - val_accuracy: 0.5664 - val_auc: 0.6803 - val_loss: 0.6435
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 28s 31ms/step - accuracy: 0.6233 - auc: 0.6757 - loss: 0.6490 - val_accuracy: 0.6504 - val_auc: 0.7049 - val_loss: 0.6241
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [11]:
# ==========================================================
# BLOCK 18 & 19: SLOWFAST MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Concatenate,
    MaxPooling3D, AveragePooling3D, Lambda
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for SlowFast.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define SlowFast Building Blocks
# ----------------------------------------------------------
def slowfast_block(x_slow, x_fast, filters, strides=(1,1,1)):
    # --- SLOW PATH (Spatial focus, 1x3x3) ---
    ys = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_slow)
    ys = BatchNormalization()(ys)
    ys = Activation('relu')(ys)
    
    ys = Conv3D(filters, (1,3,3), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(ys)
    ys = BatchNormalization()(ys)
    ys = Activation('relu')(ys)
    
    ys = Conv3D(filters*4, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(ys)
    ys = BatchNormalization()(ys)

    if strides != (1,1,1) or x_slow.shape[-1] != filters*4:
        shortcut_s = Conv3D(filters*4, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_slow)
        shortcut_s = BatchNormalization()(shortcut_s)
    else:
        shortcut_s = x_slow
        
    # --- FAST PATH (Factorized Spatiotemporal Focus) ---
    fast_filters = max(1, filters // 4)  # Kept ratio tighter for Micro scale
    
    yf = Conv3D(fast_filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_fast)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    # Factorized (3,3,3) into (1,3,3) -> (3,1,1) to prevent XLA OOM crash
    yf = Conv3D(fast_filters, (1,3,3), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    yf = Conv3D(fast_filters, (3,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    yf = Conv3D(fast_filters*4, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    
    if strides != (1,1,1) or x_fast.shape[-1] != fast_filters*4:
        shortcut_f = Conv3D(fast_filters*4, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_fast)
        shortcut_f = BatchNormalization()(shortcut_f)
    else:
        shortcut_f = x_fast

    # --- LATERAL CONNECTION ---
    ys = Add()([ys, shortcut_s])
    ys = Activation('relu')(ys)
    
    yf = Add()([yf, shortcut_f])
    yf = Activation('relu')(yf)
    
    return ys, yf

def create_slowfast_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # --- Input Splitting ---
    x_fast = video_input
    x_slow = Lambda(lambda x: x[:, ::4, :, :, :], name='slow_slice')(video_input)
    
    # --- Stem ---
    # Slow Stem
    x_slow = Conv3D(16, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(x_slow)
    x_slow = BatchNormalization()(x_slow)
    x_slow = Activation('relu')(x_slow)
    x_slow = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x_slow)
    
    # Fast Stem (Factorized)
    x_fast = Conv3D(4, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(x_fast)
    x_fast = BatchNormalization()(x_fast)
    x_fast = Activation('relu')(x_fast)
    x_fast = Conv3D(4, (3,1,1), strides=(1,1,1), padding='same', use_bias=False)(x_fast)
    x_fast = BatchNormalization()(x_fast)
    x_fast = Activation('relu')(x_fast)
    x_fast = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x_fast)
    
    # --- Stages ---
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 16)
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 32, strides=(1,2,2))
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 64, strides=(1,2,2))
    
    # --- Fusion & Head ---
    pool_slow = GlobalAveragePooling3D()(x_slow)
    pool_fast = GlobalAveragePooling3D()(x_fast)
    
    x = Concatenate()([pool_slow, pool_fast])
    
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='SlowFast_Micro')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_slowfast_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_slowfast_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for SlowFast.

Training Model: SlowFast_Micro...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - accuracy: 0.6669 - auc: 0.7374 - loss: 0.6358 - val_accuracy: 0.7478 - val_auc: 0.8345 - val_loss: 0.5504
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.7431 - auc: 0.8253 - loss: 0.5634 - val_accuracy: 0.7544 - val_auc: 0.8476 - val_loss: 0.5588
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.7566 - auc: 0.8371 - loss: 0.5545 - val_accuracy: 0.7699 - val_auc: 0.8587 - val_loss: 0.5294
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.7732 - auc: 0.8548 - loss: 0.5348 - val_accuracy: 0.7765 - val_auc: 0.8633 - val_loss: 0.5296
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.7853 - auc: 0.8704 - loss: 0.5168 - val_accuracy: 0.7566 - val_auc: 0.8499 - val_loss: 0.5514
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.7812 - auc: 0.8643 - loss: 0.5248 - val_accuracy: 0.75

In [12]:
# ==========================================================
# BLOCK 20 & 21: R(2+1)D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for R(2+1)D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define R(2+1)D Building Blocks
# ----------------------------------------------------------
def conv2plus1d(x, filters, strides=(1,1,1)):
    inter_filters = filters 

    spatial_strides = (1, strides[1], strides[2])
    x = Conv3D(inter_filters, (1, 3, 3), strides=spatial_strides, padding='same', 
               use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    temporal_strides = (strides[0], 1, 1)
    x = Conv3D(filters, (3, 1, 1), strides=temporal_strides, padding='same', 
               use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    return x

def r2plus1d_block(x, filters, strides=(1,1,1)):
    shortcut = x
    
    x = conv2plus1d(x, filters, strides=strides)
    x = conv2plus1d(x, filters)
    
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', 
                          use_bias=False, kernel_regularizer=l2(1e-5))(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_r2plus1d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(45, (1, 7, 7), strides=(1, 2, 2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(video_input)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv3D(64, (3, 1, 1), strides=(1, 1, 1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = r2plus1d_block(x, 64)
    x = r2plus1d_block(x, 64)
    
    x = r2plus1d_block(x, 128, strides=(2, 2, 2))
    x = r2plus1d_block(x, 128)
    
    x = r2plus1d_block(x, 256, strides=(2, 2, 2))
    x = r2plus1d_block(x, 256)
    
    x = r2plus1d_block(x, 512, strides=(2, 2, 2))
    x = r2plus1d_block(x, 512)

    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='R2Plus1D_18')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_r2plus1d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_r2plus1d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for R(2+1)D.

Training Model: R2Plus1D_18...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 83s 73ms/step - accuracy: 0.6733 - auc: 0.7261 - loss: 0.7940 - val_accuracy: 0.7080 - val_auc: 0.8229 - val_loss: 0.7287
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 65s 72ms/step - accuracy: 0.7318 - auc: 0.8021 - loss: 0.6802 - val_accuracy: 0.7434 - val_auc: 0.8334 - val_loss: 0.6584
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 65s 71ms/step - accuracy: 0.7514 - auc: 0.8260 - loss: 0.6453 - val_accuracy: 0.7190 - val_auc: 0.8434 - val_loss: 0.7163
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 64s 71ms/step - accuracy: 0.7594 - auc: 0.8389 - loss: 0.6288 - val_accuracy: 0.7434 - val_auc: 0.8475 - val_loss: 0.6675
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 64s 71ms/step - accuracy: 0.7784 - auc: 0.8543 - loss: 0.6079 - val_accuracy: 0.7765 - val_auc: 0.8578 - val_loss: 0.6082
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 64s 71ms/step - accuracy: 0.7875 - auc: 0.8666 - loss: 0.5935 - val_accuracy: 0.7611 -

In [13]:
# ==========================================================
# BLOCK 22 & 23: X3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Multiply, Reshape
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for X3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 5e-5  # X3D is lightweight, needs less decay
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define X3D Building Blocks
# ----------------------------------------------------------
def swish(x):
    return Activation(tf.nn.swish)(x)

def se_block(x, filters, ratio=0.25):
    inputs = x
    x = GlobalAveragePooling3D()(x)
    x = Reshape((1, 1, 1, filters))(x)
    
    reduced_filters = max(1, int(filters * ratio))
    x = Dense(reduced_filters, kernel_initializer='he_normal', use_bias=True)(x)
    x = swish(x)
    x = Dense(filters, kernel_initializer='he_normal', use_bias=True)(x)
    x = Activation('sigmoid')(x)
    
    return Multiply()([inputs, x])

def x3d_bottleneck(x, filters, strides=(1,1,1), expansion_ratio=2.25):
    shortcut = x
    input_filters = x.shape[-1]
    expanded_filters = int(input_filters * expansion_ratio)

    x = Conv3D(expanded_filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = Conv3D(expanded_filters, (3,3,3), strides=strides, padding='same', 
               groups=expanded_filters, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = se_block(x, expanded_filters)

    x = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)

    if strides != (1,1,1) or input_filters != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    x = Add()([x, shortcut])
    return x 

def create_x3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(24, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = BatchNormalization()(x)
    x = swish(x)
    
    x = Conv3D(24, (5,1,1), strides=(1,1,1), padding='same', groups=24, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = x3d_bottleneck(x, 24, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 24, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 48, strides=(1,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 48, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 48, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 96, strides=(2,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 192, strides=(1,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 192, expansion_ratio=2.25)
    
    x = Conv3D(432, (1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)
    
    x = GlobalAveragePooling3D()(x)
    
    x = Dense(2048, activation='relu')(x) 
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='X3D_M')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_x3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_x3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for X3D.

Training Model: X3D_M...
Epoch 1/50


2026-04-19 03:45:42.562507: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17039', 720 bytes spill stores, 720 bytes spill loads

2026-04-19 03:45:42.580545: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 24 bytes spill stores, 24 bytes spill loads

2026-04-19 03:45:42.587124: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 108 bytes spill stores, 108 bytes spill loads

2026-04-19 03:45:42.651998: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17039', 288 bytes spill stores, 288 bytes spill loads

2026-04-19 03:45:42.953212: I extern

906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.6929 - auc: 0.7458 - loss: 0.6461

2026-04-19 03:47:37.635917: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 124 bytes spill stores, 124 bytes spill loads

2026-04-19 03:47:37.682406: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 716 bytes spill stores, 716 bytes spill loads

2026-04-19 03:47:37.702992: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 28 bytes spill stores, 28 bytes spill loads

2026-04-19 03:47:37.738908: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 124 bytes spill stores, 124 bytes spill loads

2026-04-19 03:47:37.853197: I external/loc

906/906 ━━━━━━━━━━━━━━━━━━━━ 134s 117ms/step - accuracy: 0.7210 - auc: 0.7793 - loss: 0.6059 - val_accuracy: 0.7500 - val_auc: 0.8061 - val_loss: 0.5753
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 103s 113ms/step - accuracy: 0.7737 - auc: 0.8383 - loss: 0.5458 - val_accuracy: 0.7611 - val_auc: 0.8640 - val_loss: 0.5445
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 102s 113ms/step - accuracy: 0.7873 - auc: 0.8540 - loss: 0.5256 - val_accuracy: 0.7566 - val_auc: 0.8412 - val_loss: 0.5995
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 103s 113ms/step - accuracy: 0.8068 - auc: 0.8754 - loss: 0.4978 - val_accuracy: 0.8053 - val_auc: 0.8816 - val_loss: 0.5062
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 103s 113ms/step - accuracy: 0.8223 - auc: 0.8941 - loss: 0.4768 - val_accuracy: 0.8009 - val_auc: 0.8835 - val_loss: 0.6088
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 102s 113ms/step - accuracy: 0.8380 - auc: 0.9045 - loss: 0.4575 - val_accuracy: 0.7146 - val_auc: 0.8806 - val_loss: 0.6312
Epoch 7/50
906/906 ━━━━━━━━

In [14]:
# ==========================================================
# BLOCK 24 & 25: C3D (MODERNIZED) MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for C3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define C3D Building Blocks
# ----------------------------------------------------------
def c3d_block(x, filters, count):
    for _ in range(count):
        x = Conv3D(filters, (3, 3, 3), activation='relu', padding='same', 
                   use_bias=False, kernel_regularizer=l2(1e-5))(x)
        x = BatchNormalization()(x)
    return x

def create_c3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = c3d_block(video_input, 64, 1)
    x = MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 128, 1)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 256, 2)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 512, 2)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 512, 2)
    
    x = GlobalAveragePooling3D()(x)
    
    x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='C3D_Modernized')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_c3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_c3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for C3D.

Training Model: C3D_Modernized...
Epoch 1/50


2026-04-19 04:37:24.814646: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2695', 96 bytes spill stores, 96 bytes spill loads

2026-04-19 04:37:25.796808: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-19 04:37:25.889293: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.6518 - auc: 0.7184 - loss: 0.7881

2026-04-19 04:40:18.126540: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_327', 64 bytes spill stores, 64 bytes spill loads

2026-04-19 04:40:18.517667: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-19 04:40:18.610816: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


906/906 ━━━━━━━━━━━━━━━━━━━━ 186s 186ms/step - accuracy: 0.6774 - auc: 0.7404 - loss: 0.7690 - val_accuracy: 0.7301 - val_auc: 0.8142 - val_loss: 0.9391
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 164s 181ms/step - accuracy: 0.7227 - auc: 0.7862 - loss: 0.7241 - val_accuracy: 0.7412 - val_auc: 0.8199 - val_loss: 0.7076
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 164s 181ms/step - accuracy: 0.7503 - auc: 0.8151 - loss: 0.6938 - val_accuracy: 0.7301 - val_auc: 0.8415 - val_loss: 0.6748
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 163s 180ms/step - accuracy: 0.7685 - auc: 0.8349 - loss: 0.6710 - val_accuracy: 0.6991 - val_auc: 0.8251 - val_loss: 0.7192
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 164s 180ms/step - accuracy: 0.7613 - auc: 0.8324 - loss: 0.6693 - val_accuracy: 0.7190 - val_auc: 0.8304 - val_loss: 0.8808
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 164s 181ms/step - accuracy: 0.7790 - auc: 0.8487 - loss: 0.6507 - val_accuracy: 0.7721 - val_auc: 0.8568 - val_loss: 0.6628
Epoch 7/50
906/906 ━━━━━━━━

In [15]:
# ==========================================================
# BLOCK 26 & 27: CNN-TRANSFORMER HYBRID TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for CNN-Transformer Hybrid.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.01  # Transformers need higher decay
LABEL_SMOOTHING = 0.1
EMBED_DIM = 128      # Dimension for Transformer
NUM_HEADS = 4
TRANSFORMER_LAYERS = 2

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Model Components
# ----------------------------------------------------------
def create_cnn_feature_extractor(input_shape):
    cnn_input = layers.Input(shape=input_shape)
    
    # Stem
    x = layers.Conv2D(32, (7, 7), strides=2, padding='same', use_bias=False)(cnn_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((3, 3), strides=2, padding='same')(x)
    
    # Residual Block 1
    shortcut = x
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != 64:
        shortcut = layers.Conv2D(64, (1, 1), padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    # Residual Block 2 (Downsample)
    shortcut = x
    x = layers.Conv2D(128, (3, 3), strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    shortcut = layers.Conv2D(128, (1, 1), strides=2, padding='same', use_bias=False)(shortcut)
    shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    # Spatial Aggregation
    x = layers.GlobalAveragePooling2D()(x)
    
    return models.Model(inputs=cnn_input, outputs=x, name="cnn_extractor")

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim
        )
        self.sequence_length = sequence_length
        self.output_dim = output_dim

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        embedded_positions = self.position_embeddings(positions)
        return inputs + embedded_positions

def create_cnn_transformer_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- 1. Spatial Feature Extraction (CNN) ---
    cnn_extractor = create_cnn_feature_extractor(input_shape[1:]) 
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input) 
    
    # --- 2. Temporal Modeling (Transformer) ---
    x = layers.Dense(EMBED_DIM)(encoded_frames)
    x = PositionalEmbedding(sequence_length=NUM_FRAMES, output_dim=EMBED_DIM)(x)
    
    for _ in range(TRANSFORMER_LAYERS):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS, key_dim=EMBED_DIM, dropout=0.1
        )(x1, x1)
        x2 = layers.Add()([attention_output, x]) 

        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = layers.Dense(EMBED_DIM * 2, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(0.1)(x3)
        x3 = layers.Dense(EMBED_DIM)(x3)
        x = layers.Add()([x3, x2]) 

    # --- 3. Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    return models.Model(inputs=video_input, outputs=output, name="CNN_Transformer_Hybrid")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_cnn_transformer_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_cnn_transformer_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024)

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for CNN-Transformer Hybrid.

Training Model: CNN_Transformer_Hybrid...
Epoch 1/50


2026-04-19 06:40:50.213063: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17', 192 bytes spill stores, 192 bytes spill loads

2026-04-19 06:40:50.718308: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17_0', 184 bytes spill stores, 184 bytes spill loads

2026-04-19 06:40:50.773458: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17_0', 40 bytes spill stores, 48 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 28ms/step - accuracy: 0.6095 - auc: 0.6558 - loss: 0.7494 - val_accuracy: 0.6726 - val_auc: 0.7716 - val_loss: 0.5994
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.6736 - auc: 0.7474 - loss: 0.6423 - val_accuracy: 0.7102 - val_auc: 0.8165 - val_loss: 0.5790
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.7050 - auc: 0.7808 - loss: 0.6160 - val_accuracy: 0.7434 - val_auc: 0.8271 - val_loss: 0.5665
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.7205 - auc: 0.8023 - loss: 0.5942 - val_accuracy: 0.7389 - val_auc: 0.8278 - val_loss: 0.5759
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.7351 - auc: 0.8159 - loss: 0.5796 - val_accuracy: 0.7699 - val_auc: 0.8385 - val_loss: 0.5441
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.7464 - auc: 0.8322 - loss: 0.5631 - val_accuracy: 0.7434 - val_auc: 0.8341 - val_loss: 0.5495
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [16]:
# ==========================================================
# BLOCK 28 & 29: CONVLSTM HYBRID MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ConvLSTM.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ConvLSTM Components
# ----------------------------------------------------------
def create_cnn_backbone(input_shape):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(512, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    return models.Model(inputs=inputs, outputs=x, name="cnn_backbone")

def create_convlstm_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    cnn = create_cnn_backbone(input_shape[1:])
    x = layers.TimeDistributed(cnn)(video_input)
    
    x = layers.ConvLSTM2D(
        filters=64, 
        kernel_size=(3, 3), 
        padding='same', 
        return_sequences=False, 
        dropout=0.2,
        recurrent_dropout=0.0 
    )(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="ConvLSTM_Hybrid")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_convlstm_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_convlstm_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ConvLSTM.

Training Model: ConvLSTM_Hybrid...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 69s 61ms/step - accuracy: 0.6658 - auc: 0.7272 - loss: 0.6379 - val_accuracy: 0.7611 - val_auc: 0.8216 - val_loss: 0.5742
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.7354 - auc: 0.8139 - loss: 0.5723 - val_accuracy: 0.7611 - val_auc: 0.8381 - val_loss: 0.5414
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.7682 - auc: 0.8461 - loss: 0.5421 - val_accuracy: 0.7699 - val_auc: 0.8507 - val_loss: 0.5308
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.7850 - auc: 0.8622 - loss: 0.5242 - val_accuracy: 0.7721 - val_auc: 0.8584 - val_loss: 0.5270
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.7875 - auc: 0.8658 - loss: 0.5203 - val_accuracy: 0.7655 - val_auc: 0.8423 - val_loss: 0.5402
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.7961 - auc: 0.8742 - loss: 0.5101 - val_accuracy: 0.7

In [17]:
# ==========================================================
# BLOCK 30 & 31: TRANSFER LEARNING (MOBILENETV2 + LSTM) & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Transfer Learning.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-5 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Transfer Learning Model
# ----------------------------------------------------------
def create_transfer_mobilenet_lstm(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- The Backbone (ImageNet Pre-trained) ---
    base_cnn = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # FREEZE the backbone (Critical for Transfer Learning)
    base_cnn.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='mobilenet_feature_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input)
    
    # --- Temporal Modeling (LSTM) ---
    x = layers.LSTM(256, return_sequences=False, dropout=0.3)(encoded_frames)
    
    # --- Classification Head ---
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="TL_MobileNet_LSTM")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_transfer_mobilenet_lstm(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_tl_mobilenet_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Transfer Learning.

Training Model: TL_MobileNet_LSTM...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 72s 59ms/step - accuracy: 0.7710 - auc: 0.8424 - loss: 0.5518 - val_accuracy: 0.7765 - val_auc: 0.8819 - val_loss: 0.5614
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.8187 - auc: 0.8969 - loss: 0.4827 - val_accuracy: 0.7965 - val_auc: 0.8851 - val_loss: 0.4970
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.8308 - auc: 0.9094 - loss: 0.4646 - val_accuracy: 0.8164 - val_auc: 0.8958 - val_loss: 0.4937
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.8419 - auc: 0.9237 - loss: 0.4425 - val_accuracy: 0.8053 - val_auc: 0.8971 - val_loss: 0.4838
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.8444 - auc: 0.9264 - loss: 0.4381 - val_accuracy: 0.8053 - val_auc: 0.8901 - val_loss: 0.5060
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.8590 - auc: 0.9356 - loss: 0.4226 - val_ac

In [18]:
# ==========================================================
# BLOCK 32 & 33: TRANSFER LEARNING (RESNET50 + ATTENTION) & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResNet50 Transfer.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 
ATTENTION_HEADS = 4

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Transfer Learning Model
# ----------------------------------------------------------
def create_resnet_attention_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- The Backbone (ResNet50 - Heavyweight) ---
    base_cnn = ResNet50(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # FREEZE the backbone
    base_cnn.trainable = False
    
    # Pooling immediately to save memory (2048 features per frame)
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='resnet_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input) 
    
    # --- Temporal Modeling (Attention) ---
    x = layers.LayerNormalization(epsilon=1e-6)(encoded_frames)
    
    attention_output = layers.MultiHeadAttention(
        num_heads=ATTENTION_HEADS, 
        key_dim=2048 // ATTENTION_HEADS, 
        dropout=0.1
    )(x, x)
    
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # --- Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="TL_ResNet50_Attention")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnet_attention_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_tl_resnet_attn_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResNet50 Transfer.

Training Model: TL_ResNet50_Attention...
Epoch 1/50


2026-04-19 07:42:09.535929: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 232 bytes spill stores, 232 bytes spill loads

2026-04-19 07:42:09.566663: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 240 bytes spill stores, 240 bytes spill loads

2026-04-19 07:42:09.595275: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 560 bytes spill stores, 560 bytes spill loads

2026-04-19 07:42:09.673748: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_6_0', 40 bytes spill stores, 64 bytes spill loads

2026-04-19 07:42:09.704252: I external/lo

906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5159 - auc: 0.5210 - loss: 1.1478

2026-04-19 07:43:18.727884: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 12 bytes spill stores, 12 bytes spill loads

2026-04-19 07:43:19.003029: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 104 bytes spill stores, 104 bytes spill loads

2026-04-19 07:43:19.145586: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 3876 bytes spill stores, 3868 bytes spill loads

2026-04-19 07:43:19.188468: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 4024 bytes spill stores, 4004 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 117s 81ms/step - accuracy: 0.5041 - auc: 0.5029 - loss: 0.8573 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7261
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.5039 - auc: 0.4946 - loss: 0.7242 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7194
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.4975 - auc: 0.4888 - loss: 0.7184 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7223
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.5036 - auc: 0.4955 - loss: 0.7155 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7088
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.4903 - auc: 0.4906 - loss: 0.7087 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7047
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.4914 - auc: 0.4954 - loss: 0.7113 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7034
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━

In [19]:
# ==========================================================
# BLOCK 34 & 35: FINE-TUNED DENSENET121 + BiLSTM & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Fine-Tuning.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
FINE_TUNE_LR = 1e-5 
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Fine-Tuning Model
# ----------------------------------------------------------
def create_finetuned_densenet_bilstm(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- Backbone: DenseNet121 (ImageNet) ---
    base_cnn = DenseNet121(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # --- FINE-TUNING STRATEGY ---
    base_cnn.trainable = False
    
    # Unfreeze the last 50 layers
    for layer in base_cnn.layers[-50:]: 
        layer.trainable = True
        
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='densenet_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input)
    
    # --- Temporal Modeling: Bidirectional LSTM ---
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=False, dropout=0.3))(encoded_frames)
    
    # --- Classification Head ---
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="DenseNet_BiLSTM_FineTuned")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_finetuned_densenet_bilstm(input_shape)

optimizer = optimizers.AdamW(learning_rate=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_finetuned_densenet_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Fine-Tuning.

Training Model: DenseNet_BiLSTM_FineTuned...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 261s 208ms/step - accuracy: 0.5977 - auc: 0.6351 - loss: 0.6924 - val_accuracy: 0.7566 - val_auc: 0.8284 - val_loss: 0.5891
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 167s 184ms/step - accuracy: 0.7001 - auc: 0.7674 - loss: 0.6224 - val_accuracy: 0.7522 - val_auc: 0.8570 - val_loss: 0.5437
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 167s 184ms/step - accuracy: 0.7470 - auc: 0.8164 - loss: 0.5826 - val_accuracy: 0.7965 - val_auc: 0.8783 - val_loss: 0.5230
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 167s 184ms/step - accuracy: 0.7511 - auc: 0.8275 - loss: 0.5716 - val_accuracy: 0.7920 - val_auc: 0.8898 - val_loss: 0.5105
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 167s 184ms/step - accuracy: 0.7724 - auc: 0.8505 - loss: 0.5496 - val_accuracy: 0.7987 - val_auc: 0.8966 - val_loss: 0.4996
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 167s 184ms/step - accuracy: 0.7831 - auc: 0.8648 - loss: 0

In [4]:
# ==========================================================
# BLOCK 36 & 37: DENSENEXT3D-MICRO TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Concatenate,
    MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for DenseNeXt3D-Micro.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 5e-5  # Lower decay for micro models
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define DenseNeXt3D-Micro Building Blocks
# ----------------------------------------------------------
def swish(x):
    return Activation(tf.nn.swish)(x)

def grouped_conv3d(x, filters, kernel_size, strides=(1,1,1), padding='same', groups=4):
    """ResNeXt-style grouped convolutions for high throughput."""
    try:
        return Conv3D(filters, kernel_size, strides=strides, padding=padding, 
                      groups=groups, use_bias=False, kernel_regularizer=l2(1e-5))(x)
    except:
        group_list = []
        channels_per_group = filters // groups
        splits = tf.split(x, groups, axis=-1)
        for i in range(groups):
            g = Conv3D(channels_per_group, kernel_size, strides=strides, padding=padding, 
                       use_bias=False, kernel_regularizer=l2(1e-5))(splits[i])
            group_list.append(g)
        return Concatenate(axis=-1)(group_list)

def dense_next_layer(x, growth_rate, groups=4):
    """Combines DenseNet feature reuse with ResNeXt grouped efficiency."""
    # Bottleneck to compress channels
    x1 = Conv3D(growth_rate * 2, (1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x1 = BatchNormalization()(x1)
    x1 = swish(x1)
    
    # Grouped Spatiotemporal Convolution
    x1 = grouped_conv3d(x1, growth_rate, (3,3,3), groups=groups)
    x1 = BatchNormalization()(x1)
    x1 = swish(x1)
    
    # Dense Connection
    return Concatenate()([x, x1])

def transition_layer(x, reduction_factor=0.5):
    """Compresses the network and downsamples spatially."""
    reduced_filters = max(8, int(x.shape[-1] * reduction_factor))
    x = Conv3D(reduced_filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = swish(x)
    # Pool spatially, preserve temporal resolution
    x = MaxPooling3D((1, 2, 2), strides=(1, 2, 2), padding='same')(x) 
    return x

def create_densenext3d_micro(input_shape):
    video_input = Input(shape=input_shape)
    
    # --- Factorized Stem (SlowFast / X3D style) ---
    x = Conv3D(16, (1, 3, 3), strides=(1, 2, 2), padding='same', use_bias=False)(video_input)
    x = BatchNormalization()(x)
    x = swish(x)
    x = Conv3D(16, (3, 1, 1), strides=(1, 1, 1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)
    x = MaxPooling3D((1, 2, 2), strides=(1, 2, 2), padding='same')(x)

    # --- DenseNeXt Block 1 ---
    x = dense_next_layer(x, growth_rate=16, groups=4)
    x = dense_next_layer(x, growth_rate=16, groups=4)
    x = transition_layer(x, reduction_factor=0.5)

    # --- DenseNeXt Block 2 ---
    x = dense_next_layer(x, growth_rate=32, groups=4)
    x = dense_next_layer(x, growth_rate=32, groups=4)
    x = transition_layer(x, reduction_factor=0.5)

    # --- DenseNeXt Block 3 ---
    x = dense_next_layer(x, growth_rate=64, groups=4)
    x = dense_next_layer(x, growth_rate=64, groups=4)
    x = transition_layer(x, reduction_factor=0.5)

    # --- DenseNeXt Block 4 ---
    x = dense_next_layer(x, growth_rate=128, groups=4)
    x = dense_next_layer(x, growth_rate=128, groups=4)

    # --- Fusion & Head ---
    x = GlobalAveragePooling3D()(x)
    
    # C3D-style robust dense layer before output
    x = Dense(128, kernel_regularizer=l2(1e-4))(x)
    x = swish(x)
    x = Dropout(0.5)(x)
    
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='DenseNeXt3D_Micro')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_densenext3d_micro(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_densenext3d_micro.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for DenseNeXt3D-Micro.


I0000 00:00:1777188749.774503 2388753 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13607 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: DenseNeXt3D_Micro...
Epoch 1/50


I0000 00:00:1777188754.143905 2388975 service.cc:152] XLA service 0x7b89202161b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777188754.143927 2388975 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-26 13:32:34.337630: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777188755.162125 2388975 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-26 13:32:36.058515: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8302', 316 bytes spill stores, 316 bytes spill loads

2026-04-26 13:32:36.130546: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusio

  5/906 ━━━━━━━━━━━━━━━━━━━━ 23s 26ms/step - accuracy: 0.1342 - auc: 0.0613 - loss: 1.0542     

2026-04-26 13:32:43.150454: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion', 220 bytes spill stores, 220 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion_1', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1777188763.199896 2388975 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 38s 28ms/step - accuracy: 0.7329 - auc: 0.7974 - loss: 0.6146 - val_accuracy: 0.7235 - val_auc: 0.7833 - val_loss: 0.6512
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 24s 26ms/step - accuracy: 0.7701 - auc: 0.8405 - loss: 0.5659 - val_accuracy: 0.7854 - val_auc: 0.8760 - val_loss: 0.5518
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 24s 26ms/step - accuracy: 0.8005 - auc: 0.8710 - loss: 0.5308 - val_accuracy: 0.7301 - val_auc: 0.8574 - val_loss: 0.6603
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 24s 26ms/step - accuracy: 0.8220 - auc: 0.8902 - loss: 0.5042 - val_accuracy: 0.8075 - val_auc: 0.8805 - val_loss: 0.5244
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 24s 26ms/step - accuracy: 0.8223 - auc: 0.8984 - loss: 0.4909 - val_accuracy: 0.8208 - val_auc: 0.9106 - val_loss: 0.5131
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 24s 26ms/step - accuracy: 0.8449 - auc: 0.8999 - loss: 0.4845 - val_accuracy: 0.8584 - val_auc: 0.9126 - val_loss: 0.4721
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 36 & 37: NANO3D MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D Architecture
# ----------------------------------------------------------
def se_block_3d(x, filters, squeeze_ratio=0.25):
    """Squeeze-and-Excitation for attention-driven channel weighting."""
    squeeze_channels = max(1, int(filters * squeeze_ratio))
    se = layers.GlobalAveragePooling3D()(x)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Dense(squeeze_channels, activation=tf.nn.swish, use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([x, se])

def nano_block(x, filters, strides=(1,1,1)):
    """Factorized Spatiotemporal Block with SE & Residual Connection."""
    shortcut = x
    
    # Spatial feature extraction
    x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Temporal feature extraction
    x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Channel Attention
    x = se_block_3d(x, filters)
    
    # Residual matching
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    return x

def create_nano3d_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Stem ---
    x = layers.Conv3D(16, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Conv3D(16, (3,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.MaxPooling3D((1,2,2), strides=(1,2,2), padding='same')(x)
    
    # --- Hierarchical Blocks ---
    # Block 1 (Low-level features)
    b1 = nano_block(x, 32)
    b1 = nano_block(b1, 32)
    
    # Block 2 (Mid-level features)
    b2 = nano_block(b1, 64, strides=(1,2,2))
    b2 = nano_block(b2, 64)
    
    # Block 3 (High-level features)
    b3 = nano_block(b2, 128, strides=(2,2,2))
    b3 = nano_block(b3, 128)
    
    # --- Multi-Scale Feature Fusion (DenseNet replacement) ---
    pool1 = layers.GlobalAveragePooling3D()(b1)
    pool2 = layers.GlobalAveragePooling3D()(b2)
    pool3 = layers.GlobalAveragePooling3D()(b3)
    
    merged_features = layers.Concatenate()([pool1, pool2, pool3])
    
    # --- Classification Head ---
    x = layers.Dropout(0.4)(merged_features)
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D.

Training Model: Nano3D...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 35s 26ms/step - accuracy: 0.7299 - auc: 0.7983 - loss: 0.6116 - val_accuracy: 0.7876 - val_auc: 0.8624 - val_loss: 0.6079
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.7660 - auc: 0.8357 - loss: 0.5717 - val_accuracy: 0.8031 - val_auc: 0.8822 - val_loss: 0.5270
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.7809 - auc: 0.8578 - loss: 0.5475 - val_accuracy: 0.7810 - val_auc: 0.8979 - val_loss: 0.5271
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.8038 - auc: 0.8679 - loss: 0.5314 - val_accuracy: 0.8186 - val_auc: 0.8989 - val_loss: 0.5253
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.8107 - auc: 0.8770 - loss: 0.5180 - val_accuracy: 0.8230 - val_auc: 0.8834 - val_loss: 0.5217
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 22s 24ms/step - accuracy: 0.8256 - auc: 0.8871 - loss: 0.5036 - val_accuracy: 0.8119 - val_a

In [ ]:
# ==========================================================
# BLOCK 36 & 37: NANO3D_EDGE TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D_Edge.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D_Edge Architecture
# ----------------------------------------------------------
def se_block_3d(x, filters, squeeze_ratio=0.25):
    """Squeeze-and-Excitation (Extremely low parameters)."""
    squeeze_channels = max(1, int(filters * squeeze_ratio))
    se = layers.GlobalAveragePooling3D()(x)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Dense(squeeze_channels, activation=tf.nn.swish, use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([x, se])

def nano_edge_block(x, filters, strides=(1,1,1)):
    """Ultra-lightweight factorized block using pseudo-depthwise logic."""
    shortcut = x
    input_filters = x.shape[-1]
    
    # Spatial Pointwise Expansion (1x1x1)
    x = layers.Conv3D(filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Factorized Spatial Filtering (1x3x3) - Using groups to simulate depthwise
    groups_spatial = min(filters, 8) # Fallback to grouped if full depthwise isn't supported efficiently
    
    # Try grouped conv; if it fails (older TF), fall back to standard factorized
    try:
        x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', 
                          groups=groups_spatial, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
        x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', 
                          use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
                          
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Factorized Temporal Filtering (3x1x1)
    try:
        x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', 
                          groups=groups_spatial, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
         x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', 
                          use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
                          
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Channel Attention
    x = se_block_3d(x, filters)
    
    # Residual matching
    if strides != (1,1,1) or input_filters != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    return x

def create_nano3d_edge_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Micro Stem ---
    x = layers.Conv3D(8, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.MaxPooling3D((1,2,2), strides=(1,2,2), padding='same')(x)
    
    # --- Flat Hierarchical Blocks ---
    # Block 1 (8 -> 16 channels)
    b1 = nano_edge_block(x, 16)
    
    # Block 2 (16 -> 24 channels)
    b2 = nano_edge_block(b1, 24, strides=(1,2,2))
    
    # Block 3 (24 -> 32 channels)
    b3 = nano_edge_block(b2, 32, strides=(2,2,2))
    
    # Block 4 (32 -> 48 channels)
    b4 = nano_edge_block(b3, 48, strides=(2,2,2))
    
    # --- Multi-Scale Feature Fusion ---
    # Global average pooling on multi-scale outputs
    pool2 = layers.GlobalAveragePooling3D()(b2)
    pool3 = layers.GlobalAveragePooling3D()(b3)
    pool4 = layers.GlobalAveragePooling3D()(b4)
    
    # Concatenate features (24 + 32 + 48 = 104 parameters fed to final head)
    merged_features = layers.Concatenate()([pool2, pool3, pool4])
    
    # --- Direct Classification Head (No dense layer bottleneck) ---
    x = layers.Dropout(0.4)(merged_features)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D_Edge')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_edge_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_edge_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.4f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")


GPU Memory cleared for Nano3D_Edge.

Training Model: Nano3D_Edge...
Epoch 1/50
  8/906 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.6264 - auc: 0.6791 - loss: 0.6791 

2026-04-26 15:30:29.417933: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion', 16 bytes spill stores, 16 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion_1', 16 bytes spill stores, 16 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 31s 22ms/step - accuracy: 0.7238 - auc: 0.7978 - loss: 0.5814 - val_accuracy: 0.6438 - val_auc: 0.8486 - val_loss: 0.7950
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.7381 - auc: 0.8138 - loss: 0.5681 - val_accuracy: 0.7765 - val_auc: 0.8431 - val_loss: 0.5604
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.7580 - auc: 0.8326 - loss: 0.5483 - val_accuracy: 0.7987 - val_auc: 0.8833 - val_loss: 0.4957
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.7646 - auc: 0.8390 - loss: 0.5438 - val_accuracy: 0.7898 - val_auc: 0.8834 - val_loss: 0.5011
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.7809 - auc: 0.8590 - loss: 0.5206 - val_accuracy: 0.7743 - val_auc: 0.8756 - val_loss: 0.5471
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.7906 - auc: 0.8683 - loss: 0.5118 - val_accuracy: 0.8230 - val_auc: 0.8913 - val_loss: 0.5038
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================================
# BLOCK 38 & 39: RESFORMER3D_MAX TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResFormer3D_Max.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ResFormer3D_Max Architecture
# ----------------------------------------------------------
def res_block_3d(x, filters, strides=(1, 1, 1)):
    """Standard robust 3D Residual Block."""
    shortcut = x

    x = layers.Conv3D(filters, (3, 3, 3), strides=strides, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv3D(filters, (3, 3, 3), strides=(1, 1, 1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)

    if strides != (1, 1, 1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1, 1, 1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def transformer_encoder(inputs, embed_dim, num_heads, ff_dim, dropout=0.3):
    """Standard Transformer Encoder Block."""
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=embed_dim, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Add()([x, inputs])

    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(ff_dim, activation=tf.nn.gelu)(y)
    y = layers.Dropout(dropout)(y)
    y = layers.Dense(embed_dim)(y)
    return layers.Add()([y, x])

def create_resformer_max(input_shape):
    video_input = layers.Input(shape=input_shape)

    # --- Deep 3D CNN Backbone ---
    x = layers.Conv3D(64, (5, 5, 5), strides=(1, 2, 2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = res_block_3d(x, 64)
    x = res_block_3d(x, 128, strides=(2, 2, 2))
    x = res_block_3d(x, 256, strides=(2, 2, 2))
    x = res_block_3d(x, 512, strides=(2, 2, 2))

    # --- Prepare for Transformer ---
    # We pool spatially but retain the temporal dimension
    x = layers.AveragePooling3D(pool_size=(1, x.shape[2], x.shape[3]))(x)
    
    # Reshape to (Batch, Time, Features) for Attention
    # Note: If NUM_FRAMES=16 and we downsampled time by factor of 8, Time=2
    time_steps = x.shape[1] 
    features = x.shape[-1]
    x = layers.Reshape((time_steps, features))(x)

    # --- Multi-Head Attention Bottleneck ---
    x = transformer_encoder(x, embed_dim=512, num_heads=8, ff_dim=1024, dropout=0.4)
    x = transformer_encoder(x, embed_dim=512, num_heads=8, ff_dim=1024, dropout=0.4)

    # --- Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='ResFormer3D_Max')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resformer_max(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resformer_max.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResFormer3D_Max.

Training Model: ResFormer3D_Max...
Epoch 1/50


2026-04-26 16:24:57.679033: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_49', 112 bytes spill stores, 112 bytes spill loads

2026-04-26 16:24:57.800445: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_46', 436 bytes spill stores, 436 bytes spill loads

2026-04-26 16:24:57.898931: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_46', 188 bytes spill stores, 172 bytes spill loads

2026-04-26 16:24:57.906652: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_49', 12 bytes spill stores, 12 bytes spill loads

2026-04-26 16:24:58.519402: I external/loc

905/906 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6083 - auc: 0.6628 - loss: 1.1970

2026-04-26 16:25:52.999698: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1003', 96 bytes spill stores, 96 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 65s 53ms/step - accuracy: 0.6228 - auc: 0.6798 - loss: 1.1673 - val_accuracy: 0.7942 - val_auc: 0.8576 - val_loss: 0.7390
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.6860 - auc: 0.7441 - loss: 0.9955 - val_accuracy: 0.7743 - val_auc: 0.8446 - val_loss: 0.8014
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 51ms/step - accuracy: 0.7053 - auc: 0.7784 - loss: 0.8942 - val_accuracy: 0.7743 - val_auc: 0.8787 - val_loss: 0.7418
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 50ms/step - accuracy: 0.7431 - auc: 0.8130 - loss: 0.8096 - val_accuracy: 0.7345 - val_auc: 0.8552 - val_loss: 0.7649
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 50ms/step - accuracy: 0.7652 - auc: 0.8359 - loss: 0.7628 - val_accuracy: 0.7699 - val_auc: 0.8499 - val_loss: 0.7302
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 51ms/step - accuracy: 0.7914 - auc: 0.8609 - loss: 0.7120 - val_accuracy: 0.8274 - val_auc: 0.8921 - val_loss: 0.6803
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [4]:
# ==========================================================
# BLOCK 45 & 46: NANO3D_TURBO TRAINING & EVALUATION (REV 2)
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D_Turbo (Rev 2).")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D_Turbo Architecture
# ----------------------------------------------------------
def turbo_block(x, filters, strides=(1,1,1), groups=4):
    """Ultra-fast parallel block utilizing Grouped Convolutions."""
    shortcut = x
    
    # 1x1x1 bottleneck to reduce channel dimensions before the grouped conv
    bottleneck_filters = filters // 2
    x = layers.Conv3D(bottleneck_filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # 3x3x3 Grouped Convolution
    try:
        x = layers.Conv3D(bottleneck_filters, (3,3,3), strides=strides, padding='same', 
                          groups=groups, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
        group_list = []
        channels_per_group = bottleneck_filters // groups
        splits = tf.split(x, groups, axis=-1)
        for i in range(groups):
            g = layers.Conv3D(channels_per_group, (3,3,3), strides=strides, padding='same', 
                              use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(splits[i])
            group_list.append(g)
        x = layers.Concatenate(axis=-1)(group_list)

    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x) 
    
    # 1x1x1 expansion
    x = layers.Conv3D(filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)

    # Residual matching
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def create_nano3d_turbo_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Ultra-Fast Stem ---
    # Aggressive downsampling (2,2,2) right at the start
    x = layers.Conv3D(32, (3,3,3), strides=(2,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # --- Straight-Through Hierarchical Blocks ---
    x = turbo_block(x, 64, strides=(1,2,2), groups=4)
    x = turbo_block(x, 128, strides=(2,2,2), groups=8)
    x = turbo_block(x, 256, strides=(2,2,2), groups=16)
    
    # --- Fully Convolutional Head ---
    # No dense layers. We use a 1x1x1 conv to drop channels down to 1, then pool.
    x = layers.Conv3D(128, (1,1,1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.GlobalAveragePooling3D()(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D_Turbo')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_turbo_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_turbo_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

# Warm-up run 
_ = model.predict(test_generator[0][0], verbose=0)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D_Turbo (Rev 2).


I0000 00:00:1779038109.911913  801466 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13185 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: Nano3D_Turbo...
Epoch 1/50


I0000 00:00:1779038112.811487  801606 service.cc:152] XLA service 0x74fa0423b920 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779038112.811501  801606 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-05-17 23:15:12.917321: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779038113.446695  801606 cuda_dnn.cc:529] Loaded cuDNN version 91900


  7/906 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.4054 - auc: 0.4619 - loss: 0.7507        

I0000 00:00:1779038118.162006  801606 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 28s 22ms/step - accuracy: 0.7481 - auc: 0.8143 - loss: 0.5710 - val_accuracy: 0.7323 - val_auc: 0.8189 - val_loss: 0.5791
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.7682 - auc: 0.8440 - loss: 0.5415 - val_accuracy: 0.7456 - val_auc: 0.8434 - val_loss: 0.5608
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.7895 - auc: 0.8619 - loss: 0.5226 - val_accuracy: 0.8142 - val_auc: 0.8813 - val_loss: 0.5014
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.8082 - auc: 0.8767 - loss: 0.5039 - val_accuracy: 0.7677 - val_auc: 0.8750 - val_loss: 0.5351
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.8057 - auc: 0.8804 - loss: 0.4988 - val_accuracy: 0.8009 - val_auc: 0.8630 - val_loss: 0.5265
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.8121 - auc: 0.8906 - loss: 0.4865 - val_accuracy: 0.8252 - val_auc: 0.8876 - val_loss: 0.5055
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━